# 00 — Setup, API Preflight & Dataset Caching (`DIAL-Skill`)

Run this notebook **once** per environment (or after updating dependencies):
1. Mount Google Drive and sync `when2speak` from GitHub.
2. Install `dial_skill` dependencies and clone `deepnote/agent-initiative-benchmark`.
3. Verify `JEV_API_KEY.txt` and `OPENAI_KEY.txt` from Drive secrets (never committed!).
4. Probe OpenAI (`Luna`/`Terra`/`Sol`) + 1-item Jev health check (`< $0.01`) -> `resolved_models.json`.
5. Cache **When2Speak** and **PageMantle (~10 GB)** onto Google Drive once.

In [1]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent DRIVE_ROOT: /content/drive/MyDrive/when2speak_research


In [8]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    !pip install -q -e . --no-deps

import importlib, sys, types

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


Already up to date.
/content/when2speak
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dial-skill (pyproject.toml) ... done
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 1: Install Full Dependencies & Clone Deepnote Benchmark Repo

In [3]:
if IN_COLAB:
    !pip install -q -e ".[dev]"

EXTERNAL_BENCH_DIR = REPO_DIR / "external" / "agent-initiative-benchmark"
if not EXTERNAL_BENCH_DIR.exists():
    EXTERNAL_BENCH_DIR.parent.mkdir(parents=True, exist_ok=True)
    !git clone --depth 1 https://github.com/deepnote/agent-initiative-benchmark.git {EXTERNAL_BENCH_DIR}
else:
    !git -C {EXTERNAL_BENCH_DIR} pull --ff-only
print("Deepnote benchmark repo ready at:", EXTERNAL_BENCH_DIR)


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dial-skill (pyproject.toml) ... done
Already up to date.
Deepnote benchmark repo ready at: /content/when2speak/external/agent-initiative-benchmark


In [4]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


Secret lookup diagnostics: {
  "jev": {
    "status": "missing",
    "source": "missing",
    "masked": "<missing>"
  },
  "openai": {
    "status": "missing",
    "source": "missing",
    "masked": "<missing>"
  }
}
Resolved models: {'timestamp_utc': '2026-11-20T12:55:56.534322+00:00', 'luna': 'gpt-6-luna', 'terra': 'gpt-6.1-terra', 'sol': 'gpt-6.1-sol', 'jev_model': 'jev-latest', 'jev_healthy': False, 'triage_backend': 'openai_fallback', 'available_openai_models_count': 0, 'notes': ['OpenAI /v1/models listing failed (RuntimeError: OpenAI API key not found. Set OPENAI_API_KEY or provide OPENAI_KEY.txt.); using configured defaults.', "Probe failed for luna candidate 'gpt-6-luna': RuntimeError", "Probe failed for luna candidate 'gpt-5.6-luna': RuntimeError", "Probe failed for luna candidate 'gpt-4.1-mini': RuntimeError", "Probe failed for terra candidate 'gpt-6.1-terra': RuntimeError", "Probe failed for terra candidate 'gpt-5.6-terra': RuntimeError", "Probe failed for terra candidate 'g

## Step 2: Run Live API Preflight (`< $0.01`) -> `resolved_models.json`

In [9]:
import asyncio
from dial_skill.llm.preflight import run_preflight

RUN_LIVE_PREFLIGHT = True  # Set False for offline dry-check

if RUN_LIVE_PREFLIGHT:
    resolved = await run_preflight(models_cfg=MODELS_CFG, drive_root=DRIVE_ROOT, probe_chat_completion=True)
    print("Saved resolved_models.json:")
    print(resolved.model_dump_json(indent=2))
else:
    print("Skipped live preflight.")


CancelledError: 

## Step 3: Download & Cache When2Speak + PageMantle Datasets onto Google Drive

In [ ]:
from huggingface_hub import snapshot_download
from dial_skill.benchmarks.when2speak import load_when2speak_split

DOWNLOAD_DATASETS = True  # Downloads once to Google Drive; subsequent runs reuse Drive cache

if DOWNLOAD_DATASETS:
    # 1. Cache deterministic When2Speak splits (val_calib=200, pilot=400, main=2000)
    w2s_dir = DRIVE_ROOT / "datasets" / "when2speak"
    for preset in ["val_calib", "pilot", "main"]:
        turns = load_when2speak_split(preset, local_dir=w2s_dir, seed=EXP_CFG["seed"])
        pos = sum(1 for t in turns if t.label_speak)
        print(f"When2Speak [{preset}]: {len(turns)} turns ({pos} SPEAK = {100*pos/max(1,len(turns)):.1f}%)")

    # 2. Cache Deepnote PageMantle 11-service company dataset (~10 GB)
    pm_dir = DRIVE_ROOT / "datasets" / "pagemantle"
    if not any(pm_dir.iterdir()):
        print("Downloading deepnote/pagemantle-company-dataset to Drive (one-time ~10 GB)...")
        snapshot_download(
            repo_id=EXP_CFG["pagemantle"]["dataset_repo"],
            repo_type="dataset",
            local_dir=str(pm_dir),
        )
    else:
        print("PageMantle dataset already cached on Drive:", pm_dir)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


finetune_train_dialogue.jsonl: reconstructing file:   0%|          |  0.00B /  215MB            

finetune_train_dialogue.jsonl: downloading bytes:           |  0.00B            

finetune_test_dialogue.jsonl: reconstructing file:   0%|          |  0.00B / 26.9MB            

finetune_test_dialogue.jsonl: downloading bytes:           |  0.00B            

finetune_val_dialogue.jsonl: reconstructing file:   0%|          |  0.00B / 27.0MB            

... (3 outputs left out)

Generating validation split:   0%|          | 0/21788 [00:00<?, ? examples/s]

When2Speak [val_calib]: 200 turns (20 SPEAK = 10.0%)
When2Speak [pilot]: 400 turns (51 SPEAK = 12.8%)
When2Speak [main]: 2000 turns (249 SPEAK = 12.4%)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 31 files:   0%|          | 0/31 [00:00<?, ?it/s]

## Step 4: Offline Unit Test Sanity Check

In [ ]:
!pytest -v


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/when2speak
configfile: pyproject.toml
testpaths: tests
plugins: asyncio-1.5.0, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 30 items                                                             

tests/test_benchmarks.py ...                                             [ 10%]
tests/test_clients_and_cache.py .......                                  [ 33%]
tests/test_core_pipeline.py ......                                       [ 53%]
tests/test_evolution_dbcse.py ......                                     [ 73%]
tests/test_metrics_and_baselines.py ....                                 [ 86%]
tests/test_report_and_notebooks.py ....                                  [100%]

=======